# **Objetivo del notebook**

El presente notebook tiene como finalidad realizar la identificación, caracterización y análisis de datos atípicos (outliers), evaluando su impacto sobre la distribución de las variables, la estructura subyacente de los datos y las relaciones estadísticas presentes en el conjunto de información.

Para ello, se implementarán y compararán diferentes metodologías de detección de observaciones atípicas, abarcando tanto enfoques clásicos como robustos, desde perspectivas univariadas y multivariadas. Adicionalmente, se analizará el efecto de dichas observaciones sobre medidas descriptivas, patrones de asociación, supuestos estadísticos y procesos posteriores de modelamiento, con el fin de establecer criterios técnicamente fundamentados para su tratamiento dentro del flujo de análisis.

## **Librerías**

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import miceforest as mf
import statsmodels.api as sm
import pandas as pd
from plotly.subplots import make_subplots

## **Cargar datos**

In [31]:
# ============================================================
# CARGA DE DATOS
# ============================================================

ruta = "../data/processed/nacimientos_bogota_preprocesado.csv"

df = pd.read_csv(
    ruta,
    sep=";",
    encoding="ISO-8859-1"
)

# ============================================================
# VERIFICACIÓN INICIAL
# ============================================================

print(f"Dimensiones: {df.shape}")
df.head()

C:\Users\Usuario\AppData\Local\Temp\ipykernel_7948\1729600655.py:7: DtypeWarning:

Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.



Dimensiones: (384321, 49)


,COD_LOCALIDAD_NACIMIENTO,LOCALIDAD_NACIMIENTO,SIT_PARTO,SEXO,PESO_GRAMOS,PESO,BPN,TALLA_CENTIMETROS,ANO,ATENCION_PARTO_POR,...,SEXO_AGRUPADO,MULTIPLICIDAD_AGRUPADA,PERTENENCIA_ETNICA_AGRUPADA,TIPO_DOC_MADRE_AGRUPADO,ESTADO_CONYUGAL_AGRUPADO,NIVEL_EDUCATIVO_MADRE_AGRUPADO,NIVEL_EDUCATIVO_PADRE_AGRUPADO,REGIMEN_SEGURIDAD_AGRUPADO,APGAR1_NUM,APGAR1_AGRUPADO
0,13,Teusaquillo,1.INSTITUCION DE SALUD,MASCULINO,3330.0,5.NORMAL(3000_4199),0,51,2020,1.MEDICO,...,MASCULINO,Simple,No pertenece a grupo etnico reconocido,Colombiana identificada,Casada,Media,Media,Contributivo,8.0,Alto
1,8,Kennedy,1.INSTITUCION DE SALUD,FEMENINO,2625.0,4.DEFICIT(2500_2999),0,47,2020,1.MEDICO,...,FEMENINO,Simple,No pertenece a grupo etnico reconocido,Extranjera,Sin pareja,Media,Media,No asegurado,8.0,Alto
2,9,Fontibón,1.INSTITUCION DE SALUD,FEMENINO,3150.0,5.NORMAL(3000_4199),0,49,2020,1.MEDICO,...,FEMENINO,Simple,No pertenece a grupo etnico reconocido,Colombiana identificada,Sin pareja,Basica,Media,Subsidiado,9.0,Alto
3,2,Chapinero,1.INSTITUCION DE SALUD,MASCULINO,4475.0,6.EXCESO(MAS4200),0,54,2020,1.MEDICO,...,MASCULINO,Simple,No pertenece a grupo etnico reconocido,Colombiana identificada,Union libre,Superior,Superior,Contributivo,9.0,Alto
4,12,Barrios Unidos,1.INSTITUCION DE SALUD,FEMENINO,2380.0,3.BAJO(1500_2499),1,48,2020,1.MEDICO,...,FEMENINO,Simple,No pertenece a grupo etnico reconocido,Colombiana identificada,Union libre,Media,Media,Contributivo,8.0,Alto


## **Caracterización de outliers vía tukey**

Se realizó una evaluación univariada de valores atípicos sobre las principales variables cuantitativas del estudio utilizando el método de Tukey basado en rango intercuartílico (IQR). Para cada variable se calcularon el primer cuartil (Q1), el tercer cuartil (Q3) y el IQR (`Q3 - Q1`), definiendo como límites de atipicidad:

* Límite inferior: `Q1 - 1.5 × IQR`
* Límite superior: `Q3 + 1.5 × IQR`

A partir de estos límites, cada observación fue clasificada en una de las siguientes categorías:

* `T`: valor típico,
* `AI`: atípico inferior,
* `AS`: atípico superior,
* `F`: valor faltante.

Posteriormente, se construyó un subconjunto (`atipic_uni`) que contiene únicamente individuos con al menos un valor atípico, permitiendo identificar tanto el número total de variables atípicas por individuo como las variables específicas donde ocurrió la atipicidad.

Finalmente, se generó una caracterización global por variable, cuantificando el número y proporción de valores típicos, faltantes, atípicos inferiores y atípicos superiores. Este análisis permitió evaluar la magnitud y distribución de los valores extremos antes de aplicar procedimientos de imputación y winsorización.

In [32]:
# ============================================================
# CLASIFICACIÓN Y CARACTERIZACIÓN DE VALORES ATÍPICOS
# Método de Tukey: Q1 - 1.5*IQR / Q3 + 1.5*IQR
# ============================================================
variables_cuantitativas= ['PESO_GRAMOS',  'NUM_CONSULTAS_PRENAT',
                          'EDAD_MADRE', 'ULTIMO_CURSO_APROBADO_MADRE',  'NUM_HIJOS_NACIDOS_VIVOS',
                          'NUM_EMBARAZOS', 'EDAD_PADRE', 'ULTIMO_CURSO_APROBADO_PADRE', 'TIEMPO_GESTACION', 'TALLA_CENTIMETROS'] 

# Conservar solo variables existentes
variables_cuantitativas = [
    v for v in variables_cuantitativas
    if v in df.columns
]

# ============================================================
#  CREAR CATEGORÍAS DE ATIPICIDAD
# F  = faltante
# AI = atípico inferior
# AS = atípico superior
# T  = típico
# ============================================================

df_1 = df.copy()

for variable in variables_cuantitativas:

    serie = pd.to_numeric(df_1[variable], errors="coerce")

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    condiciones = [
        serie.isna(),
        serie < limite_inferior,
        serie > limite_superior
    ]

    valores = ["F", "AI", "AS"]

    df_1[f"{variable}_categoria"] = np.select(
        condiciones,
        valores,
        default="T"
    )

# ============================================================
#  DATAFRAME DE OBSERVACIONES CON AL MENOS UN ATÍPICO
# ============================================================

cat_cols = [f"{v}_categoria" for v in variables_cuantitativas]

mask_atipico = df_1[cat_cols].isin(["AI", "AS"]).any(axis=1)

atipic_uni = df_1.loc[mask_atipico].copy()

# Número de variables atípicas por fila
atipic_uni["n_atipicos"] = (
    atipic_uni[cat_cols]
    .isin(["AI", "AS"])
    .sum(axis=1)
)

# Lista de variables atípicas por fila
def listar_atipicos(row):
    return ", ".join(
        f"{v}({row[f'{v}_categoria']})"
        for v in variables_cuantitativas
        if row[f"{v}_categoria"] in ["AI", "AS"]
    )

atipic_uni["atipico_en"] = atipic_uni.apply(
    listar_atipicos,
    axis=1
)

# Orden final intercalado variable + categoría
orden_intercalado = ["pk_id_anom"]

for v in variables_cuantitativas:
    orden_intercalado.extend([v, f"{v}_categoria"])

orden_intercalado += ["n_atipicos", "atipico_en"]

orden_intercalado = [
    c for c in orden_intercalado
    if c in atipic_uni.columns
]

atipic_uni = (
    atipic_uni[orden_intercalado]
    .reset_index(drop=True)
)

# ============================================================
#  CARACTERIZACIÓN GLOBAL DE ATÍPICOS POR VARIABLE
# ============================================================

resumen_atipicos = []

for variable in variables_cuantitativas:

    col_cat = f"{variable}_categoria"
    conteos = df_1[col_cat].value_counts(dropna=False)

    n_ai = conteos.get("AI", 0)
    n_as = conteos.get("AS", 0)
    n_f = conteos.get("F", 0)
    n_t = conteos.get("T", 0)

    total = len(df_1)
    total_atipicos = n_ai + n_as

    resumen_atipicos.append({
        "variable": variable,
        "tipicos": n_t,
        "faltantes": n_f,
        "atipicos_inferiores": n_ai,
        "atipicos_superiores": n_as,
        "total_atipicos": total_atipicos,
        "prop_atipicos_inferiores": round(n_ai / total, 3),
        "prop_atipicos_superiores": round(n_as / total, 3),
        "prop_atipicos_total": round(total_atipicos / total, 3),
        "prop_faltantes": round(n_f / total, 3)
    })

df_caracterizacion = (
    pd.DataFrame(resumen_atipicos)
    .sort_values("total_atipicos", ascending=False)
    .reset_index(drop=True)
)

# ============================================================
#  RESULTADOS
# ============================================================

print("Total registros en df:", df.shape[0])
print("Registros con al menos un atípico:", atipic_uni.shape[0])
print("Variables evaluadas:", len(variables_cuantitativas))

#display(df_caracterizacion)
#display(atipic_uni)

# ============================================================
# DATASET SIN ATÍPICOS
# Elimina registros con al menos un AI o AS
# Conserva faltantes (F)
# ============================================================

mask_sin_atipicos = ~(
    df_1[cat_cols]
    .isin(["AI", "AS"])
    .any(axis=1)
)

df_sin_atipicos = (
    df_1.loc[mask_sin_atipicos]
    .copy()
)

print("\n======================================")
print("DATASET SIN ATÍPICOS")
print("======================================")
print("Registros originales:", len(df_1))
print("Registros eliminados:", len(df_1) - len(df_sin_atipicos))
print("Registros finales:", len(df_sin_atipicos))
print(
    "Porcentaje eliminado:",
    round(
        (len(df_1) - len(df_sin_atipicos))
        / len(df_1) * 100,
        2
    ),
    "%"
)

print("Total registros en df sin atípicos:", df_sin_atipicos.shape[0])

Total registros en df: 384321
Registros con al menos un atípico: 72055
Variables evaluadas: 10

DATASET SIN ATÍPICOS
Registros originales: 384321
Registros eliminados: 72055
Registros finales: 312266
Porcentaje eliminado: 18.75 %
Total registros en df sin atípicos: 312266


### Tratamiento robusto de variables cuantitativas para modelamiento del peso al nacer

Este bloque implementa un proceso sistemático de depuración, imputación y control de valores extremos sobre las variables cuantitativas utilizadas en el análisis. El objetivo principal es mejorar la calidad de los datos antes del modelamiento estadístico, preservando la coherencia clínica de las variables y reduciendo la influencia de registros atípicos potencialmente derivados de errores de captura, digitación o codificación administrativa.

Inicialmente se identifican las variables cuantitativas de interés y se convierten explícitamente a formato numérico. Posteriormente, diversos códigos administrativos utilizados para representar información desconocida o no disponible (por ejemplo, valores como 99 o 9999) son recodificados como valores faltantes (`NaN`) para evitar que sean interpretados erróneamente como observaciones reales.

La variable objetivo del estudio (`PESO_GRAMOS`) se excluye deliberadamente de cualquier procedimiento de imputación o winsorización, con el fin de preservar íntegramente la distribución observada del desenlace.

A continuación, para cada variable explicativa cuantitativa se calculan límites de detección de valores extremos utilizando la regla de Tukey basada en el rango intercuartílico (IQR):

$$
IQR = Q_3 - Q_1
$$

$$
LI = Q_1 - 1.5 \times IQR
$$

$$
LS = Q_3 + 1.5 \times IQR
$$

donde (Q_1) y (Q_3) corresponden al primer y tercer cuartil respectivamente.

Para variables de conteo, los límites obtenidos se ajustan a valores enteros mediante redondeo superior, garantizando consistencia con la naturaleza discreta de estas mediciones.

Los valores faltantes de las variables explicativas son posteriormente imputados utilizando la mediana observada de cada variable, una estrategia robusta frente a distribuciones asimétricas y presencia de valores extremos.

Después de la imputación se aplica un proceso de winsorización. Este procedimiento reemplaza observaciones ubicadas fuera de los límites establecidos por Tukey por el valor del límite correspondiente. Para variables con asimetría positiva esperada (por ejemplo, número de consultas prenatales o número de embarazos) únicamente se restringe la cola superior de la distribución, mientras que para el resto de variables se controlan ambas colas.

Adicionalmente se generan indicadores auxiliares (*flags*) que registran la ocurrencia de valores faltantes y observaciones originalmente ubicadas fuera de los límites definidos. Estos indicadores permiten mantener trazabilidad sobre las transformaciones realizadas y facilitan posteriores análisis de sensibilidad o auditoría de calidad de datos.

Finalmente, se construyen resúmenes diagnósticos que documentan los límites utilizados, la cantidad de observaciones imputadas por variable, el porcentaje de datos faltantes tratado y la presencia residual de valores fuera de rango tras el proceso de limpieza. El resultado es un conjunto de datos cuantitativos depurado, consistente y preparado para etapas posteriores de análisis exploratorio y modelamiento predictivo.


In [33]:

# ============================================================
# VARIABLES CUANTITATIVAS
# ============================================================

variables_cuantitativas = [
    "PESO_GRAMOS",
    "NUM_CONSULTAS_PRENAT",
    "EDAD_MADRE",
    "ULTIMO_CURSO_APROBADO_MADRE",
    "NUM_HIJOS_NACIDOS_VIVOS",
    "NUM_EMBARAZOS",
    "EDAD_PADRE",
    "ULTIMO_CURSO_APROBADO_PADRE",
    "TIEMPO_GESTACION",
    "TALLA_CENTIMETROS"
]

variables_cuantitativas = [
    v for v in variables_cuantitativas
    if v in df.columns
]

# ============================================================
# VARIABLE OBJETIVO
# NO SE IMPUTA NI SE WINSORIZA
# ============================================================

variable_objetivo = "PESO_GRAMOS"

# ============================================================
# VARIABLES CON COLA DERECHA ESPERADA
# ============================================================

asimetria = {
    "NUM_CONSULTAS_PRENAT": "up",
    "NUM_HIJOS_NACIDOS_VIVOS": "up",
    "NUM_EMBARAZOS": "up"
}

# ============================================================
# VARIABLES DE CONTEO
# ============================================================

variables_conteo = [
    "NUM_CONSULTAS_PRENAT",
    "NUM_HIJOS_NACIDOS_VIVOS",
    "NUM_EMBARAZOS"
]

# ============================================================
# CÓDIGOS ADMINISTRATIVOS A TRATAR COMO NA
# ============================================================

codigos_na = {
    "ULTIMO_CURSO_APROBADO_MADRE": [99],
    "ULTIMO_CURSO_APROBADO_PADRE": [99],
    "PESO_GRAMOS": [9999],
    "TALLA_CENTIMETROS": [99],
    "NUM_HIJOS_NACIDOS_VIVOS": [99],
    "NUM_CONSULTAS_PRENAT": [99],
    "NUM_EMBARAZOS": [99],
}

# ============================================================
# COPIA DE TRABAJO
# ============================================================

df_base = df.copy()

# ============================================================
# CONVERSIÓN A NUMÉRICO
# ============================================================

for col in variables_cuantitativas:

    df_base[col] = pd.to_numeric(
        df_base[col],
        errors="coerce"
    )

# ============================================================
# REEMPLAZAR CÓDIGOS ADMINISTRATIVOS POR NaN
# ============================================================

for col, valores in codigos_na.items():

    if col not in df_base.columns:
        continue

    df_base[col] = df_base[col].replace(
        valores,
        np.nan
    )

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def calcular_limites_tukey(
    df,
    variables,
    k=1.5,
    asimetria=None
):

    registros = []

    for col in variables:

        serie = (
            pd.to_numeric(df[col], errors="coerce")
            .dropna()
        )

        modo = (asimetria or {}).get(
            col,
            "both"
        )

        if len(serie) == 0:

            registros.append({
                "variable": col,
                "q1": np.nan,
                "q3": np.nan,
                "iqr": np.nan,
                "limite_inferior": np.nan,
                "limite_superior": np.nan,
                "mediana_imputacion": np.nan,
                "asimetria": modo
            })

            continue

        q1 = serie.quantile(0.25)
        q3 = serie.quantile(0.75)

        iqr = q3 - q1

        registros.append({
            "variable": col,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "limite_inferior": q1 - k * iqr,
            "limite_superior": q3 + k * iqr,
            "mediana_imputacion": serie.median(),
            "asimetria": modo
        })

    return (
        pd.DataFrame(registros)
        .set_index("variable")
    )


def ajustar_limites_conteo(
    limites,
    variables_conteo
):

    limites = limites.copy()

    for col in variables_conteo:

        if col not in limites.index:
            continue

        limites.loc[col, "limite_inferior"] = np.ceil(
            limites.loc[col, "limite_inferior"]
        )

        limites.loc[col, "limite_superior"] = np.ceil(
            limites.loc[col, "limite_superior"]
        )

    return limites


def imputar_winsorizar(
    df,
    limites,
    variable_objetivo=None,
    crear_flags=True
):

    df_proc = df.copy()

    for col, row in limites.iterrows():

        if col not in df_proc.columns:
            continue

        serie_original = pd.to_numeric(
            df_proc[col],
            errors="coerce"
        )

        low = row["limite_inferior"]
        high = row["limite_superior"]
        mediana = row["mediana_imputacion"]

        modo = str(
            row["asimetria"]
        ).lower()

        # -----------------------------------
        # FLAGS
        # -----------------------------------

        if crear_flags:

            df_proc[f"{col}_missing"] = (
                serie_original.isna()
                .astype(int)
            )

            df_proc[f"{col}_out_low"] = (
                (serie_original < low)
                .astype(int)
                if np.isfinite(low)
                else 0
            )

            df_proc[f"{col}_out_high"] = (
                (serie_original > high)
                .astype(int)
                if np.isfinite(high)
                else 0
            )

        # -----------------------------------
        # VARIABLE OBJETIVO
        # NO IMPUTAR
        # NO WINSORIZAR
        # -----------------------------------

        if col == variable_objetivo:

            df_proc[col] = serie_original

            continue

        # -----------------------------------
        # IMPUTACIÓN
        # -----------------------------------

        df_proc[col] = serie_original.fillna(
            mediana
        )

        # -----------------------------------
        # WINSORIZACIÓN
        # -----------------------------------

        if modo in ["both", "down"]:

            df_proc.loc[
                df_proc[col] < low,
                col
            ] = low

        if modo in ["both", "up"]:

            df_proc.loc[
                df_proc[col] > high,
                col
            ] = high

    return df_proc


def resumen_post_winsorizacion(
    df_proc,
    limites,
    variable_objetivo=None
):

    resumen = []

    for col, row in limites.iterrows():

        serie = pd.to_numeric(
            df_proc[col],
            errors="coerce"
        )

        low = row["limite_inferior"]
        high = row["limite_superior"]

        n_low = (
            (serie < low).sum()
            if np.isfinite(low)
            else 0
        )

        n_high = (
            (serie > high).sum()
            if np.isfinite(high)
            else 0
        )

        resumen.append({
            "variable": col,
            "variable_objetivo": col == variable_objetivo,
            "fuera_limite_inferior": int(n_low),
            "fuera_limite_superior": int(n_high),
            "total_fuera_limites": int(
                n_low + n_high
            )
        })

    return pd.DataFrame(resumen)

# ============================================================
# VARIABLES A PROCESAR
# EXCLUYENDO LA VARIABLE OBJETIVO
# ============================================================

variables_modelado = [
    v
    for v in variables_cuantitativas
    if v != variable_objetivo
]

# ============================================================
# LÍMITES TUKEY
# ============================================================

limites_tukey = calcular_limites_tukey(
    df=df_base,
    variables=variables_modelado,
    k=1.5,
    asimetria=asimetria
)

limites_tukey = ajustar_limites_conteo(
    limites_tukey,
    variables_conteo
)

# ============================================================
# IMPUTACIÓN + WINSORIZACIÓN
# ============================================================

df_limpio = imputar_winsorizar(
    df=df_base,
    limites=limites_tukey,
    variable_objetivo=variable_objetivo,
    crear_flags=True
)

# ============================================================
# VALIDACIÓN
# ============================================================

resumen_post = resumen_post_winsorizacion(
    df_limpio,
    limites_tukey,
    variable_objetivo
)

print("\nLÍMITES TUKEY")
display(
    limites_tukey.round(3)
)

print("\nRESUMEN POSTERIOR")
display(resumen_post)

# ============================================================
# RESUMEN DE IMPUTACIÓN
# ============================================================

resumen_imputacion = []

for col in variables_modelado:

    n_missing = (
        df_base[col]
        .isna()
        .sum()
    )

    resumen_imputacion.append({
        "variable": col,
        "n_imputados": n_missing,
        "porcentaje_imputado":
            round(
                n_missing /
                len(df_base) * 100,
                3
            ),
        "mediana_imputacion":
            round(
                limites_tukey.loc[
                    col,
                    "mediana_imputacion"
                ],
                3
            )
    })

df_resumen_imputacion = (
    pd.DataFrame(
        resumen_imputacion
    )
    .sort_values(
        "porcentaje_imputado",
        ascending=False
    )
)

print("\nRESUMEN DE IMPUTACIÓN")
display(df_resumen_imputacion)

print("\nDimensión original:", df.shape)
print("Dimensión procesada:", df_limpio.shape)


LÍMITES TUKEY


C:\Users\Usuario\AppData\Local\Temp\ipykernel_7948\1465409735.py:258: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '9.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.



,q1,q3,iqr,limite_inferior,limite_superior,mediana_imputacion,asimetria
variable,,,,,,,
NUM_CONSULTAS_PRENAT,5.0,8.0,3.0,1.0,13.0,7.0,up
EDAD_MADRE,23.0,32.0,9.0,9.5,45.5,28.0,both
ULTIMO_CURSO_APROBADO_MADRE,4.0,11.0,7.0,-6.5,21.5,6.0,both
NUM_HIJOS_NACIDOS_VIVOS,1.0,2.0,1.0,-0.0,4.0,1.0,up
NUM_EMBARAZOS,1.0,2.0,1.0,-0.0,4.0,2.0,up
EDAD_PADRE,25.0,36.0,11.0,8.5,52.5,30.0,both
ULTIMO_CURSO_APROBADO_PADRE,5.0,11.0,6.0,-4.0,20.0,9.0,both
TIEMPO_GESTACION,37.0,39.0,2.0,34.0,42.0,38.0,both
TALLA_CENTIMETROS,48.0,51.0,3.0,43.5,55.5,50.0,both



RESUMEN POSTERIOR


,variable,variable_objetivo,fuera_limite_inferior,fuera_limite_superior,total_fuera_limites
0,NUM_CONSULTAS_PRENAT,False,16754,0,16754
1,EDAD_MADRE,False,0,0,0
2,ULTIMO_CURSO_APROBADO_MADRE,False,0,0,0
3,NUM_HIJOS_NACIDOS_VIVOS,False,0,0,0
4,NUM_EMBARAZOS,False,0,0,0
5,EDAD_PADRE,False,0,0,0
6,ULTIMO_CURSO_APROBADO_PADRE,False,0,0,0
7,TIEMPO_GESTACION,False,0,0,0
8,TALLA_CENTIMETROS,False,0,0,0



RESUMEN DE IMPUTACIÓN


,variable,n_imputados,porcentaje_imputado,mediana_imputacion
6,ULTIMO_CURSO_APROBADO_PADRE,15629,4.067,9.0
5,EDAD_PADRE,3806,0.990,30.0
4,NUM_EMBARAZOS,3279,0.853,2.0
3,NUM_HIJOS_NACIDOS_VIVOS,3275,0.852,1.0
2,ULTIMO_CURSO_APROBADO_MADRE,1354,0.352,6.0
7,TIEMPO_GESTACION,113,0.029,38.0
8,TALLA_CENTIMETROS,88,0.023,50.0
0,NUM_CONSULTAS_PRENAT,39,0.010,7.0
1,EDAD_MADRE,0,0.000,28.0



Dimensión original: (384321, 49)
Dimensión procesada: (384321, 76)


## **Análisis de sensibilidad frente al tratamiento de datos faltantes y valores extremos**

In [34]:
# ============================================================
# DESCRIPTIVOS COMPARATIVOS
# df, df_limpio y df_sin_atipicos
# ============================================================

datasets = {
    "Original": df,
    "Limpio": df_limpio,
    "Sin_atipicos": df_sin_atipicos
}

resumen_descriptivo = []

for nombre, datos in datasets.items():

    desc = (
        datos[variables_cuantitativas]
        .describe()
        .T
        .reset_index()
        .rename(columns={"index": "variable"})
    )

    desc["dataset"] = nombre

    resumen_descriptivo.append(desc)

resumen_descriptivo = pd.concat(
    resumen_descriptivo,
    ignore_index=True
)

# Reordenar columnas
cols = [
    "dataset",
    "variable",
    "count",
    "mean",
    "std",
    "min",
    "25%",
    "50%",
    "75%",
    "max"
]

resumen_descriptivo = resumen_descriptivo[cols]

display(resumen_descriptivo)

,dataset,variable,count,mean,std,min,25%,50%,75%,max
0,Original,PESO_GRAMOS,384320.0,2936.476585,507.967693,345.0,2680.0,2980.0,3255.0,9999.0
1,Original,NUM_CONSULTAS_PRENAT,384321.0,6.664057,3.074439,0.0,5.0,7.0,8.0,99.0
2,Original,EDAD_MADRE,384321.0,27.876710,6.318831,11.0,23.0,28.0,32.0,54.0
3,Original,ULTIMO_CURSO_APROBADO_MADRE,384046.0,7.331054,6.102316,0.0,4.0,6.0,11.0,99.0
4,Original,NUM_HIJOS_NACIDOS_VIVOS,381047.0,1.697463,0.899549,1.0,1.0,1.0,2.0,99.0
5,Original,NUM_EMBARAZOS,381047.0,1.916333,1.139519,1.0,1.0,2.0,2.0,99.0
6,Original,EDAD_PADRE,380515.0,30.883237,7.571711,0.0,25.0,30.0,36.0,80.0
7,Original,ULTIMO_CURSO_APROBADO_PADRE,368692.0,7.535455,3.588675,0.0,5.0,9.0,11.0,13.0
8,Original,TIEMPO_GESTACION,384208.0,38.017150,1.906626,20.0,37.0,38.0,39.0,43.0
9,Original,TALLA_CENTIMETROS,384321.0,49.218846,2.937924,17.0,48.0,50.0,51.0,99.0


In [35]:
datasets = {
    "Original": df,
    "Limpio": df_limpio,
    "Sin_atipicos": df_sin_atipicos
}

for nombre, datos in datasets.items():

    # ---------------------------------------
    # Variables numéricas disponibles
    # ---------------------------------------

    vars_disp = [
        v for v in variables_cuantitativas
        if v in datos.columns
    ]

    # ---------------------------------------
    # Correlación de Pearson
    # ---------------------------------------

    corr = (
        datos[vars_disp]
        .corr(method="pearson")
    )

    print(f"\nCorrelación de Pearson - {nombre}")
    display(corr.round(3))


Correlación de Pearson - Original


,PESO_GRAMOS,NUM_CONSULTAS_PRENAT,EDAD_MADRE,ULTIMO_CURSO_APROBADO_MADRE,NUM_HIJOS_NACIDOS_VIVOS,NUM_EMBARAZOS,EDAD_PADRE,ULTIMO_CURSO_APROBADO_PADRE,TIEMPO_GESTACION,TALLA_CENTIMETROS
PESO_GRAMOS,1.000,0.099,0.033,0.011,0.007,0.017,0.026,-0.033,0.729,0.810
NUM_CONSULTAS_PRENAT,0.099,1.000,0.304,-0.135,-0.113,-0.070,0.222,-0.252,0.092,0.043
EDAD_MADRE,0.033,0.304,1.000,-0.169,0.337,0.354,0.642,-0.273,-0.028,-0.015
ULTIMO_CURSO_APROBADO_MADRE,0.011,-0.135,-0.169,1.000,0.058,0.032,-0.146,0.424,-0.009,0.043
NUM_HIJOS_NACIDOS_VIVOS,0.007,-0.113,0.337,0.058,1.000,0.820,0.238,0.073,-0.056,0.002
NUM_EMBARAZOS,0.017,-0.070,0.354,0.032,0.820,1.000,0.248,0.039,-0.046,0.006
EDAD_PADRE,0.026,0.222,0.642,-0.146,0.238,0.248,1.000,-0.239,-0.015,-0.011
ULTIMO_CURSO_APROBADO_PADRE,-0.033,-0.252,-0.273,0.424,0.073,0.039,-0.239,1.000,-0.006,0.005
TIEMPO_GESTACION,0.729,0.092,-0.028,-0.009,-0.056,-0.046,-0.015,-0.006,1.000,0.713
TALLA_CENTIMETROS,0.810,0.043,-0.015,0.043,0.002,0.006,-0.011,0.005,0.713,1.000



Correlación de Pearson - Limpio


,PESO_GRAMOS,NUM_CONSULTAS_PRENAT,EDAD_MADRE,ULTIMO_CURSO_APROBADO_MADRE,NUM_HIJOS_NACIDOS_VIVOS,NUM_EMBARAZOS,EDAD_PADRE,ULTIMO_CURSO_APROBADO_PADRE,TIEMPO_GESTACION,TALLA_CENTIMETROS
PESO_GRAMOS,1.000,0.109,0.033,-0.026,0.010,0.024,0.035,-0.034,0.693,0.775
NUM_CONSULTAS_PRENAT,0.109,1.000,0.322,-0.264,-0.126,-0.079,0.241,-0.255,0.082,0.024
EDAD_MADRE,0.033,0.322,1.000,-0.283,0.343,0.375,0.654,-0.269,-0.033,-0.016
ULTIMO_CURSO_APROBADO_MADRE,-0.026,-0.264,-0.283,1.000,0.099,0.061,-0.204,0.485,0.003,0.021
NUM_HIJOS_NACIDOS_VIVOS,0.010,-0.126,0.343,0.099,1.000,0.847,0.252,0.072,-0.067,0.008
NUM_EMBARAZOS,0.024,-0.079,0.375,0.061,0.847,1.000,0.273,0.042,-0.056,0.014
EDAD_PADRE,0.035,0.241,0.654,-0.204,0.252,0.273,1.000,-0.239,-0.019,-0.001
ULTIMO_CURSO_APROBADO_PADRE,-0.034,-0.255,-0.269,0.485,0.072,0.042,-0.239,1.000,-0.004,0.008
TIEMPO_GESTACION,0.693,0.082,-0.033,0.003,-0.067,-0.056,-0.019,-0.004,1.000,0.614
TALLA_CENTIMETROS,0.775,0.024,-0.016,0.021,0.008,0.014,-0.001,0.008,0.614,1.000



Correlación de Pearson - Sin_atipicos


,PESO_GRAMOS,NUM_CONSULTAS_PRENAT,EDAD_MADRE,ULTIMO_CURSO_APROBADO_MADRE,NUM_HIJOS_NACIDOS_VIVOS,NUM_EMBARAZOS,EDAD_PADRE,ULTIMO_CURSO_APROBADO_PADRE,TIEMPO_GESTACION,TALLA_CENTIMETROS
PESO_GRAMOS,1.000,0.074,0.052,-0.032,0.029,0.041,0.049,-0.036,0.575,0.700
NUM_CONSULTAS_PRENAT,0.074,1.000,0.324,-0.253,-0.066,-0.036,0.248,-0.247,0.043,-0.006
EDAD_MADRE,0.052,0.324,1.000,-0.308,0.307,0.327,0.676,-0.292,-0.027,-0.009
ULTIMO_CURSO_APROBADO_MADRE,-0.032,-0.253,-0.308,1.000,0.082,0.045,-0.227,0.497,0.008,0.022
NUM_HIJOS_NACIDOS_VIVOS,0.029,-0.066,0.307,0.082,1.000,0.838,0.242,0.065,-0.064,0.013
NUM_EMBARAZOS,0.041,-0.036,0.327,0.045,0.838,1.000,0.254,0.034,-0.051,0.018
EDAD_PADRE,0.049,0.248,0.676,-0.227,0.242,0.254,1.000,-0.253,-0.019,0.003
ULTIMO_CURSO_APROBADO_PADRE,-0.036,-0.247,-0.292,0.497,0.065,0.034,-0.253,1.000,0.004,0.013
TIEMPO_GESTACION,0.575,0.043,-0.027,0.008,-0.064,-0.051,-0.019,0.004,1.000,0.487
TALLA_CENTIMETROS,0.700,-0.006,-0.009,0.022,0.013,0.018,0.003,0.013,0.487,1.000


In [36]:
# ============================================================
# ELIMINAR FLAGS Y VARIABLES AUXILIARES
# ============================================================

cols_eliminar = [
    'NUM_CONSULTAS_PRENAT_missing',
    'NUM_CONSULTAS_PRENAT_out_low',
    'NUM_CONSULTAS_PRENAT_out_high',

    'EDAD_MADRE_missing',
    'EDAD_MADRE_out_low',
    'EDAD_MADRE_out_high',

    'ULTIMO_CURSO_APROBADO_MADRE_missing',
    'ULTIMO_CURSO_APROBADO_MADRE_out_low',
    'ULTIMO_CURSO_APROBADO_MADRE_out_high',

    'NUM_HIJOS_NACIDOS_VIVOS_missing',
    'NUM_HIJOS_NACIDOS_VIVOS_out_low',
    'NUM_HIJOS_NACIDOS_VIVOS_out_high',

    'NUM_EMBARAZOS_missing',
    'NUM_EMBARAZOS_out_low',
    'NUM_EMBARAZOS_out_high',

    'EDAD_PADRE_missing',
    'EDAD_PADRE_out_low',
    'EDAD_PADRE_out_high',

    'ULTIMO_CURSO_APROBADO_PADRE_missing',
    'ULTIMO_CURSO_APROBADO_PADRE_out_low',
    'ULTIMO_CURSO_APROBADO_PADRE_out_high',

    'TIEMPO_GESTACION_missing',
    'TIEMPO_GESTACION_out_low',
    'TIEMPO_GESTACION_out_high',

    'TALLA_CENTIMETROS_missing',
    'TALLA_CENTIMETROS_out_low',
    'TALLA_CENTIMETROS_out_high',

    'APGAR1_NUM',
    'BPN_NUM'
]

df_limpio = df_limpio.drop(
    columns=cols_eliminar,
    errors="ignore"
)

print("Dimensión final:", df_limpio.shape)

Dimensión final: (384321, 47)


In [37]:
df_limpio.to_csv("../data/processed/nacimientos_bogota_preprocesado.csv", index=False, sep=";", encoding="ISO-8859-1") 